# The 10 Bugs — Interactive Debugging

**Lecture 3, Part 5 — Vizuara Modern Robot Learning Bootcamp**

Deploying a learned policy to a real robot is where **silent failures** live. Your code runs, PyTorch is happy, the robot moves — but the behavior is completely wrong. This notebook walks through the **10 most common deployment bugs** using a simulated SO-101 pipeline, so you can learn to spot and fix them before they waste hours of debugging on hardware.

| # | Bug | Category |
|---|-----|----------|
| 1 | State vector ordering | Data pipeline |
| 2 | Action chunking (no queue) | Control loop |
| 3 | `.float()` on SmolVLA | Model loading |
| 4 | Features not populated | Config |
| 5 | Missing postprocessor | Data pipeline |
| 6 | Camera rename map missing | Observation |
| 7 | Resolution mismatch | Observation |
| 8 | Episode too short | Control loop |
| 9 | `torch.load` serialization | Checkpointing |
| 10 | Wrong import path | Setup |

**Every bug in this list is silent** — your code runs without errors, but the robot does the wrong thing (or does nothing useful).

---

In [ ]:
# Install dependencies (Colab already has these, but just in case)
!pip install -q torch numpy matplotlib 2>&1 | tail -3

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import deque, OrderedDict
import copy

# Vizuara color palette
ACCENT = '#D97757'
BLUE   = '#5B8CB8'
TEAL   = '#7DA488'
PURPLE = '#9B7EC8'
WARM   = '#C4956A'
RED    = '#D4543A'
BG     = '#FAF8F5'

plt.rcParams['figure.facecolor'] = BG
plt.rcParams['axes.facecolor'] = BG
plt.rcParams['font.size'] = 12

torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

---

## Step 1: Setup — The Simulated SO-101 Pipeline

We build a simulated robot with 6 joints. Every bug in this notebook uses this same pipeline, so you only need to understand it once.

In [ ]:
# Joint names in the CORRECT order (as the robot hardware reports them)
JOINT_NAMES = ["shoulder_pan", "shoulder_lift", "elbow_flex",
               "wrist_flex", "wrist_roll", "gripper"]

# Normalization stats (realistic values for SO-101)
NORM_STATS = {
    "shoulder_pan":   {"mean": 0.0,   "std": 45.0},
    "shoulder_lift":  {"mean": -30.0, "std": 25.0},
    "elbow_flex":     {"mean": 90.0,  "std": 35.0},
    "wrist_flex":     {"mean": 0.0,   "std": 40.0},
    "wrist_roll":     {"mean": 0.0,   "std": 60.0},
    "gripper":        {"mean": 50.0,  "std": 30.0},
}

print("Joint names (CORRECT hardware order):")
for i, name in enumerate(JOINT_NAMES):
    stats = NORM_STATS[name]
    print(f"  [{i}] {name:>15s}  mean={stats['mean']:>6.1f}  std={stats['std']:>5.1f}")

In [ ]:
class SimulatedRobot:
    """Simulated SO-101 arm for debugging exercises."""

    def __init__(self, joint_names=JOINT_NAMES, norm_stats=NORM_STATS):
        self.joint_names = joint_names
        self.norm_stats = norm_stats
        self.n_joints = len(joint_names)

    def generate_episode(self, n_steps=300, freq_range=(0.5, 2.0)):
        """Generate smooth sinusoidal joint trajectories for one episode."""
        t = np.linspace(0, 2 * np.pi, n_steps)
        trajectories = {}
        for name in self.joint_names:
            freq = np.random.uniform(*freq_range)
            phase = np.random.uniform(0, 2 * np.pi)
            amplitude = self.norm_stats[name]["std"] * 0.8
            mean = self.norm_stats[name]["mean"]
            trajectories[name] = mean + amplitude * np.sin(freq * t + phase)
        return trajectories

    def read_state_dict(self, trajectories, step):
        """Read observation state as a dictionary at a given timestep."""
        return {name: trajectories[name][step] for name in self.joint_names}

    def state_dict_to_vector(self, state_dict, order):
        """Convert state dict to vector using the given joint order."""
        return np.array([state_dict[name] for name in order])

    def normalize(self, values, order):
        """Normalize a vector given a joint ordering."""
        means = np.array([self.norm_stats[name]["mean"] for name in order])
        stds = np.array([self.norm_stats[name]["std"] for name in order])
        return (values - means) / stds

    def denormalize(self, normed, order):
        """Denormalize a vector given a joint ordering."""
        means = np.array([self.norm_stats[name]["mean"] for name in order])
        stds = np.array([self.norm_stats[name]["std"] for name in order])
        return normed * stds + means


robot = SimulatedRobot()
episode = robot.generate_episode(n_steps=300)

# Show a snapshot
state = robot.read_state_dict(episode, step=50)
print("State at step 50:")
for name, val in state.items():
    print(f"  {name:>15s}: {val:>8.2f} deg")

In [ ]:
# Demonstrate the core problem: "code runs, output is garbage"
correct_order = JOINT_NAMES
wrong_order = sorted(JOINT_NAMES)  # alphabetical

state = robot.read_state_dict(episode, step=50)
vec_correct = robot.state_dict_to_vector(state, correct_order)
normed_correct = robot.normalize(vec_correct, correct_order)

# Both look like "reasonable" normalized vectors
vec_wrong = robot.state_dict_to_vector(state, wrong_order)
normed_wrong = robot.normalize(vec_wrong, wrong_order)

print("CORRECT order normalized: ", [f"{v:>6.3f}" for v in normed_correct])
print("WRONG   order normalized: ", [f"{v:>6.3f}" for v in normed_wrong])
print()
print("Both look like plausible numbers between -2 and 2.")
print("No error. No warning. But the robot will do completely different things.")
print("\nThis is the 'silent failure' problem.")

---

## Step 2: Bug #1 — State Vector Order

The single most common deployment bug. The dataset recorded joints in one order; your deployment code reads them in another. No error is thrown — the normalized values all look reasonable — but the robot receives completely wrong commands.

In [ ]:
# ============================================================
# BUGGY CODE: uses sorted() which gives ALPHABETICAL order
# ============================================================

def build_state_vector_BUGGY(state_dict):
    """BUG: sorted() produces alphabetical order, not hardware order!"""
    return np.array([state_dict[k] for k in sorted(state_dict.keys())])

# What alphabetical order looks like:
alpha_order = sorted(JOINT_NAMES)
print("CORRECT hardware order:", JOINT_NAMES)
print("WRONG alphabetical order:", alpha_order)
print()
print("Mapping confusion:")
for i, (correct, wrong) in enumerate(zip(JOINT_NAMES, alpha_order)):
    match = 'OK' if correct == wrong else 'SWAPPED'
    print(f"  slot [{i}]: should be {correct:>15s}, but gets {wrong:>15s}  [{match}]")

In [ ]:
# Visualize the catastrophic effect
state = robot.read_state_dict(episode, step=50)
raw_values = [state[name] for name in JOINT_NAMES]  # ground truth

# Normalize with CORRECT order, then denormalize with CORRECT order → perfect
vec_correct = robot.state_dict_to_vector(state, JOINT_NAMES)
normed = robot.normalize(vec_correct, JOINT_NAMES)
recovered_correct = robot.denormalize(normed, JOINT_NAMES)

# Normalize with WRONG order, then denormalize with CORRECT order → disaster
vec_wrong = build_state_vector_BUGGY(state)  # alphabetical order
normed_wrong = robot.normalize(vec_wrong, JOINT_NAMES)  # normalize as if hardware order
recovered_wrong = robot.denormalize(normed_wrong, JOINT_NAMES)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

x = np.arange(len(JOINT_NAMES))
width = 0.35

# Left: normalized values comparison
ax = axes[0]
ax.bar(x - width/2, normed, width, color=TEAL, label='Correct order', edgecolor='white')
ax.bar(x + width/2, normed_wrong, width, color=RED, label='Alphabetical order (BUG)', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(JOINT_NAMES, rotation=30, ha='right', fontsize=10)
ax.set_ylabel('Normalized value', fontweight='bold')
ax.set_title('Normalized Values: Correct vs Wrong Order', fontsize=13, color=ACCENT, fontweight='bold')
ax.legend(fontsize=10)
ax.axhline(y=0, color='gray', linewidth=0.5)

# Right: recovered joint angles
ax = axes[1]
ax.bar(x - width/2, recovered_correct, width, color=TEAL, label='Correct recovery', edgecolor='white')
ax.bar(x + width/2, recovered_wrong, width, color=RED, label='Wrong order recovery', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(JOINT_NAMES, rotation=30, ha='right', fontsize=10)
ax.set_ylabel('Joint angle (degrees)', fontweight='bold')
ax.set_title('Recovered Joint Angles: Correct vs Corrupted', fontsize=13, color=RED, fontweight='bold')
ax.legend(fontsize=10)
ax.axhline(y=0, color='gray', linewidth=0.5)

plt.tight_layout()
plt.show()

print("The right chart is the real danger: the WRONG values look plausible,")
print("but every joint is getting a command meant for a different joint!")
print()
for i, name in enumerate(JOINT_NAMES):
    print(f"  {name:>15s}: correct={recovered_correct[i]:>8.2f}  wrong={recovered_wrong[i]:>8.2f}  delta={abs(recovered_correct[i]-recovered_wrong[i]):>8.2f}")

**Challenge:** Can you spot what's wrong?

The bug is `sorted(state_dict.keys())` — Python dict keys are in insertion order (Python 3.7+), but `sorted()` forces alphabetical. The fix is trivially simple:

In [ ]:
# ============================================================
# FIXED CODE: use explicit hardware order, NEVER sorted()
# ============================================================

def build_state_vector_FIXED(state_dict, joint_order=JOINT_NAMES):
    """CORRECT: always use the explicit hardware joint order."""
    return np.array([state_dict[name] for name in joint_order])

# Verify the fix
vec_fixed = build_state_vector_FIXED(state)
normed_fixed = robot.normalize(vec_fixed, JOINT_NAMES)
recovered_fixed = robot.denormalize(normed_fixed, JOINT_NAMES)

print("FIXED recovery:")
for i, name in enumerate(JOINT_NAMES):
    original = raw_values[i]
    fixed = recovered_fixed[i]
    print(f"  {name:>15s}: original={original:>8.2f}  recovered={fixed:>8.2f}  error={abs(original-fixed):.6f}")
print("\nAll errors are 0.0 — perfect round-trip!")

---

## Step 3: Bug #2 — Action Chunking (Must Use Queue)

Modern policies predict a **chunk** of future actions (e.g., 50 steps). The naive approach regenerates the entire chunk every timestep and only uses the first action. This is:
1. Wasteful (running the model 50x more than needed)
2. **Jerky** (each call produces slightly different predictions due to stochasticity)

In [ ]:
# ============================================================
# BUGGY CODE: regenerate trajectory every step, use only first action
# ============================================================

def fake_model_predict(step, chunk_size=50):
    """Simulate a policy that outputs a chunk of 50 actions.
    Adds slight randomness each call to simulate real model stochasticity."""
    t = np.linspace(step * 0.02, step * 0.02 + chunk_size * 0.02, chunk_size)
    base_trajectory = np.sin(t) * 30.0  # smooth sinusoidal
    noise = np.random.randn(chunk_size) * 2.0  # stochastic noise
    return base_trajectory + noise


def bad_control_loop(n_steps=200):
    """BAD: regenerate trajectory every step, always use action[0]."""
    trajectory = []
    model_calls = 0
    for step in range(n_steps):
        chunk = fake_model_predict(step)  # 50 actions, expensive!
        model_calls += 1
        trajectory.append(chunk[0])       # always use first action
    return np.array(trajectory), model_calls


def good_control_loop(n_steps=200):
    """GOOD: use action queue, only call model when queue is empty."""
    trajectory = []
    queue = deque()
    model_calls = 0
    for step in range(n_steps):
        if len(queue) == 0:
            chunk = fake_model_predict(step)
            model_calls += 1
            queue.extend(chunk)
        trajectory.append(queue.popleft())
    return np.array(trajectory), model_calls


np.random.seed(42)
bad_traj, bad_calls = bad_control_loop(200)
np.random.seed(42)
good_traj, good_calls = good_control_loop(200)

print(f"BAD  loop: {bad_calls:>4d} model calls for {len(bad_traj)} steps")
print(f"GOOD loop: {good_calls:>4d} model calls for {len(good_traj)} steps")
print(f"\nThe BAD loop calls the model {bad_calls/good_calls:.0f}x more often!")

In [ ]:
# Visualize: jerky (BAD) vs smooth (GOOD) trajectories
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax = axes[0]
ax.plot(bad_traj, color=RED, linewidth=1.5, alpha=0.9)
ax.set_ylabel('Joint angle (deg)', fontweight='bold')
ax.set_title('BAD: Regenerate every step (jerky, 200 model calls)', fontsize=13, color=RED, fontweight='bold')
ax.axhline(y=0, color='gray', linewidth=0.5)

ax = axes[1]
ax.plot(good_traj, color=TEAL, linewidth=1.5, alpha=0.9)
# Mark where model was called
for i in range(0, len(good_traj), 50):
    if i < len(good_traj):
        ax.axvline(x=i, color=BLUE, linewidth=0.8, alpha=0.4, linestyle='--')
ax.set_ylabel('Joint angle (deg)', fontweight='bold')
ax.set_xlabel('Timestep', fontweight='bold')
ax.set_title('GOOD: Action queue (smooth, 4 model calls)', fontsize=13, color=TEAL, fontweight='bold')
ax.axhline(y=0, color='gray', linewidth=0.5)

plt.tight_layout()
plt.show()

# Compute jerkiness (second derivative)
bad_jerk = np.mean(np.abs(np.diff(bad_traj, n=2)))
good_jerk = np.mean(np.abs(np.diff(good_traj, n=2)))
print(f"Jerkiness (mean |2nd derivative|):")
print(f"  BAD:  {bad_jerk:.3f}")
print(f"  GOOD: {good_jerk:.3f}")
print(f"  BAD is {bad_jerk/good_jerk:.1f}x jerkier!")

**Challenge:** Can you spot what's wrong?

The bad loop throws away 49 out of 50 predicted actions every step. The fix: use a `deque` as an action queue and only call the model when it's empty.

---

## Step 4: Bug #3 — `.float()` on SmolVLA

SmolVLA stores weights in **bfloat16**. Calling `.float()` silently converts them to float32. The model still runs — but the outputs change because bfloat16 and float32 represent numbers differently.

In [ ]:
# ============================================================
# BUGGY CODE: calling .float() on a bfloat16 model
# ============================================================

class SimulatedSmolVLA(nn.Module):
    """A simple network simulating SmolVLA's action head."""
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(6, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 6)
        )

    def forward(self, x):
        return self.layers(x)


torch.manual_seed(42)
model_bf16 = SimulatedSmolVLA().to(torch.bfloat16)
model_bf16.eval()

# Create a copy and call .float() — simulating the common mistake
model_f32 = copy.deepcopy(model_bf16).float()  # BUG: .float() changes weights

# Same input
test_input = torch.randn(1, 6)

with torch.no_grad():
    out_bf16 = model_bf16(test_input.to(torch.bfloat16))
    out_f32 = model_f32(test_input.float())

print(f"Model dtype: bfloat16 → {model_bf16.layers[0].weight.dtype}")
print(f"Model dtype: float32  → {model_f32.layers[0].weight.dtype}")
print()
print(f"bfloat16 output: {out_bf16.float().numpy().flatten()}")
print(f"float32  output: {out_f32.numpy().flatten()}")
print()
diff = (out_bf16.float() - out_f32).abs()
print(f"Max absolute difference: {diff.max().item():.6f}")
print(f"Mean absolute difference: {diff.mean().item():.6f}")

In [ ]:
# Visualize the output differences across many inputs
torch.manual_seed(0)
n_samples = 100
diffs = []
for _ in range(n_samples):
    x = torch.randn(1, 6)
    with torch.no_grad():
        o_bf16 = model_bf16(x.to(torch.bfloat16)).float()
        o_f32 = model_f32(x.float())
    diffs.append((o_bf16 - o_f32).abs().numpy().flatten())

diffs = np.array(diffs)  # [100, 6]

fig, ax = plt.subplots(figsize=(12, 5))
bp = ax.boxplot(diffs, labels=JOINT_NAMES, patch_artist=True,
                boxprops=dict(facecolor=RED, alpha=0.6),
                medianprops=dict(color='white', linewidth=2))
ax.set_ylabel('Absolute output difference', fontweight='bold')
ax.set_title('.float() Conversion Error Per Joint (100 random inputs)', fontsize=13, color=RED, fontweight='bold')
ax.set_xlabel('Joint', fontweight='bold')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

print("These differences look small, but they compound over an episode.")
print("After 300 steps of action chunking, the trajectory can diverge significantly.")
print()
print("Rule: NEVER call .float() on a model loaded from a bfloat16 checkpoint.")
print('Instead, cast your INPUT: model(x.to(model.dtype))')

**Challenge:** Can you spot what's wrong?

The fix: instead of converting the model to float32, convert the input to the model's dtype:
```python
# BAD:
model = model.float()
output = model(input_tensor)

# GOOD:
output = model(input_tensor.to(model.dtype))
```

In [ ]:
# ============================================================
# FIXED CODE: keep model in bfloat16, cast input instead
# ============================================================

def run_model_FIXED(model, input_tensor):
    """CORRECT: cast input to model dtype, never cast model."""
    model_dtype = next(model.parameters()).dtype
    return model(input_tensor.to(model_dtype))

# Verify
test_x = torch.randn(1, 6)
with torch.no_grad():
    out_fixed = run_model_FIXED(model_bf16, test_x)
    out_direct = model_bf16(test_x.to(torch.bfloat16))

print(f"FIXED output matches direct bfloat16 call: {torch.allclose(out_fixed, out_direct)}")
print(f"Output dtype preserved: {out_fixed.dtype}")

---

## Step 5: Bug #4 — Features Not Populated

When you call `SmolVLAPolicy.from_pretrained()`, the config contains **empty** feature dictionaries. You must populate them with the dataset's actual feature metadata (image shapes, state dimensions) before the policy can build its observation preprocessing.

In [ ]:
# ============================================================
# BUGGY CODE: features not populated after from_pretrained()
# ============================================================

# Simulate the config that comes back from from_pretrained()
class FakeConfig:
    def __init__(self):
        self.input_features = {}   # EMPTY — not populated!
        self.output_features = {}  # EMPTY — not populated!
        self.image_size = 224

config_buggy = FakeConfig()
print("Config after from_pretrained():")
print(f"  input_features:  {config_buggy.input_features}")
print(f"  output_features: {config_buggy.output_features}")
print()

# Try to build observations
def build_observations_BUGGY(config, state_dict, images):
    """This will fail or produce garbage because features are empty."""
    obs = {}
    for key in config.input_features:
        if 'image' in key:
            obs[key] = images[key]
        elif 'state' in key:
            obs[key] = state_dict
    return obs

# Simulate calling it
fake_state = {"shoulder_pan": 10.0}
fake_images = {"camera1": np.zeros((224, 224, 3))}

obs = build_observations_BUGGY(config_buggy, fake_state, fake_images)
print(f"Built observations: {obs}")
print("\nThe observations dict is EMPTY — the model gets no input!")
print("No error is thrown. The model just receives an empty batch.")

In [ ]:
# ============================================================
# FIXED CODE: populate features from dataset metadata
# ============================================================

# In real code, you'd get this from the dataset's features
DATASET_FEATURES = {
    "input_features": {
        "observation.images.webcam": {"shape": [3, 224, 224], "dtype": "float32"},
        "observation.images.arm_cam": {"shape": [3, 224, 224], "dtype": "float32"},
        "observation.state": {"shape": [6], "dtype": "float32"},
    },
    "output_features": {
        "action": {"shape": [6], "dtype": "float32"},
    }
}

config_fixed = FakeConfig()
config_fixed.input_features = DATASET_FEATURES["input_features"]
config_fixed.output_features = DATASET_FEATURES["output_features"]

print("Config after populating features:")
print(f"  input_features:")
for k, v in config_fixed.input_features.items():
    print(f"    {k}: shape={v['shape']}")
print(f"  output_features:")
for k, v in config_fixed.output_features.items():
    print(f"    {k}: shape={v['shape']}")

# Now the observation builder works correctly
def build_observations_FIXED(config, state_vec, images):
    """CORRECT: features are populated, so we know what to build."""
    obs = {}
    for key, meta in config.input_features.items():
        if 'image' in key:
            cam_name = key.split('.')[-1]
            obs[key] = images.get(cam_name, np.zeros(meta['shape']))
        elif 'state' in key:
            obs[key] = state_vec
    return obs

fake_images_fixed = {
    "webcam": np.random.randn(3, 224, 224).astype(np.float32),
    "arm_cam": np.random.randn(3, 224, 224).astype(np.float32),
}
fake_state_vec = np.array([10.0, -20.0, 85.0, 5.0, 30.0, 80.0], dtype=np.float32)

obs_fixed = build_observations_FIXED(config_fixed, fake_state_vec, fake_images_fixed)
print("\nBuilt observations (FIXED):")
for k, v in obs_fixed.items():
    if isinstance(v, np.ndarray):
        print(f"  {k}: shape={v.shape}, dtype={v.dtype}")
    else:
        print(f"  {k}: {v}")
print("\nAll 3 inputs present — model can run correctly.")

---

## Step 6: Bug #5 — Missing Preprocessor/Postprocessor

The model outputs **normalized** action values (roughly in [-1, 1]). Without a postprocessor to denormalize them, these values go directly to the robot joints — which expect degrees.

In [ ]:
# ============================================================
# BUGGY CODE: raw model output sent directly to robot
# ============================================================

raw_model_output = np.array([-0.3, 0.7, -0.1, 0.5, -0.8, 0.2])

print("Raw model output (normalized):")
for name, val in zip(JOINT_NAMES, raw_model_output):
    print(f"  {name:>15s}: {val:>7.3f}")
print()
print("Without postprocessor, these values go DIRECTLY to the robot.")
print("The shoulder_pan motor receives -0.3 degrees instead of the")
print(f"intended {-0.3 * 45.0 + 0.0:.1f} degrees. The gripper gets 0.2")
print(f"instead of {0.2 * 30.0 + 50.0:.1f} degrees. Nothing works!")

In [ ]:
# ============================================================
# FIXED CODE: postprocessor denormalizes model output
# ============================================================

class Postprocessor:
    """Denormalize model outputs back to physical joint space."""
    def __init__(self, joint_names, norm_stats):
        self.means = np.array([norm_stats[n]["mean"] for n in joint_names])
        self.stds = np.array([norm_stats[n]["std"] for n in joint_names])

    def __call__(self, normalized_actions):
        return normalized_actions * self.stds + self.means


postprocessor = Postprocessor(JOINT_NAMES, NORM_STATS)
denormed_output = postprocessor(raw_model_output)

# Visualize
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(JOINT_NAMES))
width = 0.35

bars_raw = ax.bar(x - width/2, raw_model_output, width, color=RED,
                  label='Raw model output (no postprocessor)', edgecolor='white')
bars_fixed = ax.bar(x + width/2, denormed_output, width, color=TEAL,
                    label='Postprocessed (denormalized)', edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(JOINT_NAMES, rotation=25, ha='right', fontsize=10)
ax.set_ylabel('Value sent to robot', fontweight='bold')
ax.set_title('Model Output: Raw vs Postprocessed', fontsize=13, color=ACCENT, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')
ax.axhline(y=0, color='gray', linewidth=0.5)

# Annotate the bars
for bar, val in zip(bars_raw, raw_model_output):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}', ha='center', fontsize=9, color=RED)
for bar, val in zip(bars_fixed, denormed_output):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}', ha='center', fontsize=9, color=TEAL)

plt.tight_layout()
plt.show()

print("Comparison:")
for name, raw, denorm in zip(JOINT_NAMES, raw_model_output, denormed_output):
    print(f"  {name:>15s}: raw={raw:>7.3f}  denormalized={denorm:>8.2f} deg")

---

## Step 7: Bugs #6-8 — Camera Issues

Three related bugs that all involve how camera observations reach the model.

### Bug #6: Camera Rename Map Missing

Your dataset used camera names `webcam` and `arm_cam`. The model config expects `camera1` and `camera2`. Without a rename mapping, images get lost or swapped.

In [ ]:
# ============================================================
# BUGGY CODE: no rename map — camera names don't match
# ============================================================

# What the robot sends
robot_cameras = {
    "webcam": np.random.rand(224, 224, 3),     # overhead view
    "arm_cam": np.random.rand(224, 224, 3),    # wrist view
}

# What the model expects
model_expected_cameras = ["observation.images.camera1", "observation.images.camera2"]

# Without rename map: try to look up robot camera names in model format
def build_camera_obs_BUGGY(robot_cameras, expected_keys):
    """BUG: no rename map, names don't match."""
    obs = {}
    for key in expected_keys:
        cam_name = key.split('.')[-1]  # 'camera1', 'camera2'
        if cam_name in robot_cameras:
            obs[key] = robot_cameras[cam_name]
        else:
            obs[key] = np.zeros((224, 224, 3))  # silent zero fallback!
            # No warning, no error — just a black image
    return obs

obs_buggy = build_camera_obs_BUGGY(robot_cameras, model_expected_cameras)
print("BUGGY observation:")
for k, v in obs_buggy.items():
    is_zero = np.allclose(v, 0)
    print(f"  {k}: {'ALL ZEROS (black image!)' if is_zero else 'has data'}")
print("\nBoth cameras are BLACK — the model receives no visual input!")

In [ ]:
# Visualize the problem
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Overhead = blue-ish, wrist = orange-ish (for clarity)
overhead_img = np.zeros((224, 224, 3))
overhead_img[:, :, 2] = 0.6  # blue tint
overhead_img[80:140, 80:140] = [0.2, 0.3, 0.8]  # object

wrist_img = np.zeros((224, 224, 3))
wrist_img[:, :, 0] = 0.5  # red/orange tint
wrist_img[90:130, 90:160] = [0.9, 0.5, 0.2]  # gripper

axes[0].imshow(overhead_img)
axes[0].set_title('Robot: "webcam"\n(overhead)', fontsize=11, color=BLUE, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(wrist_img)
axes[1].set_title('Robot: "arm_cam"\n(wrist)', fontsize=11, color=ACCENT, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(np.zeros((224, 224, 3)))
axes[2].set_title('Model gets: "camera1"\nBLACK (missing!)', fontsize=11, color=RED, fontweight='bold')
axes[2].axis('off')

axes[3].imshow(np.zeros((224, 224, 3)))
axes[3].set_title('Model gets: "camera2"\nBLACK (missing!)', fontsize=11, color=RED, fontweight='bold')
axes[3].axis('off')

plt.suptitle('Bug #6: Without Rename Map, Model Gets Black Images',
             fontsize=14, color=RED, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# FIXED CODE: explicit rename map
# ============================================================

CAMERA_RENAME_MAP = {
    "webcam": "observation.images.camera1",    # overhead → camera1
    "arm_cam": "observation.images.camera2",   # wrist → camera2
}

def build_camera_obs_FIXED(robot_cameras, rename_map):
    """CORRECT: rename robot camera names to model's expected names."""
    obs = {}
    for robot_name, model_key in rename_map.items():
        if robot_name in robot_cameras:
            obs[model_key] = robot_cameras[robot_name]
        else:
            raise ValueError(f"Camera '{robot_name}' not found in robot output!")
    return obs

robot_cameras_real = {"webcam": overhead_img, "arm_cam": wrist_img}
obs_fixed = build_camera_obs_FIXED(robot_cameras_real, CAMERA_RENAME_MAP)

print("FIXED observation:")
for k, v in obs_fixed.items():
    is_zero = np.allclose(v, 0)
    print(f"  {k}: shape={v.shape}, has_data={not is_zero}")
print("\nBoth cameras properly mapped!")

### Bug #7: Resolution Mismatch

Setting your camera to capture at 224x224 (to match the model) seems logical. But SmolVLA expects to receive **full-resolution** images (e.g., 640x480) and resizes them internally. Capturing at 224x224 directly causes a loss of field-of-view.

In [ ]:
# Simulate the resolution mismatch
def make_scene_image(h, w):
    """Create a synthetic scene with objects at various positions."""
    img = np.ones((h, w, 3)) * 0.9  # light gray background
    # Table (brown)
    table_top = int(h * 0.6)
    img[table_top:, :] = [0.55, 0.35, 0.2]
    # Red cup (center-left)
    cy, cx = int(h * 0.45), int(w * 0.3)
    rr = int(min(h, w) * 0.08)
    for y in range(max(0, cy-rr), min(h, cy+rr)):
        for x in range(max(0, cx-rr), min(w, cx+rr)):
            if (y-cy)**2 + (x-cx)**2 < rr**2:
                img[y, x] = [0.85, 0.2, 0.15]
    # Blue plate (center-right)
    cy2, cx2 = int(h * 0.5), int(w * 0.7)
    rr2 = int(min(h, w) * 0.1)
    for y in range(max(0, cy2-rr2), min(h, cy2+rr2)):
        for x in range(max(0, cx2-rr2), min(w, cx2+rr2)):
            if (y-cy2)**2 + (x-cx2)**2 < rr2**2:
                img[y, x] = [0.2, 0.4, 0.75]
    # Green cube (far right)
    cy3, cx3 = int(h * 0.48), int(w * 0.88)
    ss = int(min(h, w) * 0.06)
    img[max(0,cy3-ss):min(h,cy3+ss), max(0,cx3-ss):min(w,cx3+ss)] = [0.2, 0.65, 0.3]
    return np.clip(img, 0, 1)

# Full resolution capture
full_res = make_scene_image(480, 640)

# Direct 224x224 capture (BAD — crops field of view)
direct_224 = make_scene_image(224, 224)

# Proper resize from full res (GOOD — preserves entire scene)
from PIL import Image as PILImage
try:
    pil_img = PILImage.fromarray((full_res * 255).astype(np.uint8))
    resized = np.array(pil_img.resize((224, 224))) / 255.0
except ImportError:
    # Fallback: simple nearest-neighbor resize
    idx_y = np.linspace(0, 479, 224).astype(int)
    idx_x = np.linspace(0, 639, 224).astype(int)
    resized = full_res[np.ix_(idx_y, idx_x)]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].imshow(full_res)
axes[0].set_title('Full resolution 640x480\n(what camera sees)', fontsize=11, color=BLUE, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(direct_224)
axes[1].set_title('Direct 224x224 capture\n(BAD: lost field-of-view!)', fontsize=11, color=RED, fontweight='bold')
axes[1].axis('off')
axes[1].text(112, 210, 'Green cube MISSING!', ha='center', fontsize=10,
             color='white', fontweight='bold', bbox=dict(boxstyle='round', facecolor=RED, alpha=0.8))

axes[2].imshow(resized)
axes[2].set_title('640x480 → resized to 224x224\n(GOOD: full scene preserved)', fontsize=11, color=TEAL, fontweight='bold')
axes[2].axis('off')

plt.suptitle('Bug #7: Capture at Full Resolution, Let Model Resize Internally',
             fontsize=14, color=ACCENT, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("The green cube on the far right is MISSING in direct 224x224 capture!")
print("Always capture at full resolution and let the model's internal transforms resize.")

### Bug #8: Episode Too Short

Setting `n_steps=50` at 30 Hz gives only 1.67 seconds. The robot barely starts moving before the episode ends.

In [ ]:
# ============================================================
# BUGGY CODE: episode too short
# ============================================================

FPS = 30
short_steps = 50
long_steps = 300

episode_data = robot.generate_episode(n_steps=long_steps)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=False)

colors = [ACCENT, BLUE, TEAL, PURPLE, WARM, RED]

ax = axes[0]
for i, name in enumerate(JOINT_NAMES):
    ax.plot(range(short_steps), episode_data[name][:short_steps],
            color=colors[i], linewidth=2, label=name)
ax.axvline(x=short_steps-1, color=RED, linewidth=2, linestyle='--', label='Episode end')
ax.set_title(f'BAD: n_steps={short_steps} ({short_steps/FPS:.2f}s at {FPS}Hz) — task barely starts!',
             fontsize=13, color=RED, fontweight='bold')
ax.set_ylabel('Joint angle (deg)', fontweight='bold')
ax.set_xlabel('Timestep', fontweight='bold')
ax.legend(fontsize=8, ncol=3, loc='upper right')
ax.set_xlim(0, long_steps)

ax = axes[1]
for i, name in enumerate(JOINT_NAMES):
    ax.plot(range(long_steps), episode_data[name][:long_steps],
            color=colors[i], linewidth=2, label=name)
ax.set_title(f'GOOD: n_steps={long_steps} ({long_steps/FPS:.1f}s at {FPS}Hz) — full task completion',
             fontsize=13, color=TEAL, fontweight='bold')
ax.set_ylabel('Joint angle (deg)', fontweight='bold')
ax.set_xlabel('Timestep', fontweight='bold')
ax.legend(fontsize=8, ncol=3, loc='upper right')

plt.tight_layout()
plt.show()

print(f"Episode length math:")
print(f"  {short_steps} steps x {1000/FPS:.0f}ms = {short_steps/FPS:.2f}s  ← TOO SHORT")
print(f"  {long_steps} steps x {1000/FPS:.0f}ms = {long_steps/FPS:.1f}s   ← RECOMMENDED MINIMUM")
print()
print("A pick-and-place task needs 8-15 seconds. 1.67 seconds is useless.")
print("Rule of thumb: start with 300-500 steps at 30Hz (10-17 seconds).")

---

## Step 8: Bugs #9-10 — Serialization + Import Paths

These bugs crash loudly (unlike the silent ones above), but they're still confusing the first time you see them.

### Bug #9: `torch.load` NormalizationMode Error

Newer versions of PyTorch default to `weights_only=True` in `torch.load()`. SmolVLA checkpoints may contain custom enum classes (like `NormalizationMode`) that aren't on the safe list, causing a crash.

In [ ]:
# ============================================================
# Simulate the torch.load serialization error
# ============================================================

import enum
import tempfile
import os

class NormalizationMode(enum.Enum):
    """Custom enum that SmolVLA checkpoints contain."""
    MEAN_STD = "mean_std"
    MIN_MAX = "min_max"

# Save a checkpoint containing the custom enum
checkpoint = {
    "model_state": {"weight": torch.randn(6, 6)},
    "normalization_mode": NormalizationMode.MEAN_STD,
    "norm_stats": {"mean": torch.zeros(6), "std": torch.ones(6)},
}

tmp_path = os.path.join(tempfile.gettempdir(), "test_checkpoint.pt")
torch.save(checkpoint, tmp_path)

# Try loading with weights_only=True (the new default)
print("Attempting torch.load with weights_only=True...")
try:
    loaded = torch.load(tmp_path, weights_only=True)
    print("Loaded successfully (older PyTorch version)")
except Exception as e:
    error_msg = str(e)
    # Show a truncated version of the error
    print(f"ERROR: {error_msg[:200]}...")
    print()
    print("This crash happens because NormalizationMode is not in the safe globals list.")

In [ ]:
# ============================================================
# FIXED CODE: add custom class to safe globals
# ============================================================

# Fix: register the custom class as safe
try:
    torch.serialization.add_safe_globals([NormalizationMode])
    loaded = torch.load(tmp_path, weights_only=True)
    print("FIXED: Loaded successfully after adding NormalizationMode to safe globals!")
    print(f"  normalization_mode: {loaded['normalization_mode']}")
    print(f"  model weight shape: {loaded['model_state']['weight'].shape}")
except AttributeError:
    # Older PyTorch without add_safe_globals
    loaded = torch.load(tmp_path, weights_only=False)
    print("FIXED (older PyTorch): Loaded with weights_only=False")
    print(f"  normalization_mode: {loaded['normalization_mode']}")
    print(f"  model weight shape: {loaded['model_state']['weight'].shape}")

# Clean up
os.remove(tmp_path)
print()
print("The fix is one line:")
print("  torch.serialization.add_safe_globals([NormalizationMode])")
print("  # Then torch.load() works with weights_only=True")

### Bug #10: `so_follower` vs `so101_follower`

A simple naming confusion that causes an import error. The correct name is `so101_follower`, not `so_follower`.

In [ ]:
# ============================================================
# Demonstrate the naming confusion
# ============================================================

wrong_names = {
    "--robot.type=so_follower":     "--robot.type=so101_follower",
    "--teleop.type=so_leader":      "--teleop.type=so101_leader",
    "from lerobot.configs.robot.so_follower": "from lerobot.configs.robot.so101_follower",
}

print("Common naming mistakes:")
print(f"{'':>4s} {'WRONG':<45s}  {'CORRECT':<45s}")
print(f"{'':>4s} {'='*45}  {'='*45}")
for wrong, correct in wrong_names.items():
    print(f"{'':>4s} {wrong:<45s}  {correct:<45s}")

print()
print("Other common confusion pairs:")
confusions = [
    ("SmolVLA", "SmolVLAPolicy", "Class name in lerobot"),
    ("/dev/ttyACM0", "/dev/tty.usbmodem*", "Serial port on macOS"),
    ("observation.state", "observation.environment_state", "LeRobot key name"),
    ("act", "act_plus_plus", "Policy name in older configs"),
]
print(f"{'':>4s} {'Wrong':<35s}  {'Correct':<35s}  {'Context':<30s}")
print(f"{'':>4s} {'-'*35}  {'-'*35}  {'-'*30}")
for wrong, correct, ctx in confusions:
    print(f"{'':>4s} {wrong:<35s}  {correct:<35s}  {ctx:<30s}")

---

## Step 9: Master Debugging Checklist + Summary

### Debugging Flowchart

```
Robot moves randomly?
 ├── Check Bug #1: State vector order (sorted() vs hardware order)
 └── Check Bug #5: Missing postprocessor (raw normalized values to motors)

Robot jerks violently?
 └── Check Bug #2: Action chunking (regenerating every step vs queue)

Model loads but outputs are wrong?
 ├── Check Bug #3: .float() converting bfloat16 weights
 └── Check Bug #4: Features not populated in config

Images look correct but model fails?
 ├── Check Bug #6: Camera rename map missing
 └── Check Bug #7: Resolution mismatch (224x224 capture vs resize)

Training works but episodes are too short?
 └── Check Bug #8: Episode length too short (50 steps = 1.67s)

torch.load crashes?
 └── Check Bug #9: NormalizationMode not in safe globals

Import error on robot type?
 └── Check Bug #10: so_follower → so101_follower
```

In [ ]:
# Full 10-bug summary visualization
bugs = [
    ("#1", "State vector order",         "Robot moves randomly",            "Use explicit JOINT_NAMES order, never sorted()",     True),
    ("#2", "Action chunking",            "Robot jerks violently",           "Use deque action queue",                              True),
    ("#3", ".float() on SmolVLA",        "Outputs subtly wrong",            "Cast input to model.dtype, not model to float32",    True),
    ("#4", "Features not populated",     "Empty observations",              "Populate from dataset metadata after from_pretrained", True),
    ("#5", "Missing postprocessor",      "Tiny values sent to motors",      "Add denormalization postprocessor",                   True),
    ("#6", "Camera rename map",          "Black/swapped images",            "Explicit rename_map: webcam->camera1",                True),
    ("#7", "Resolution mismatch",        "Lost field-of-view",             "Capture full-res, let model resize",                  True),
    ("#8", "Episode too short",          "Task never completes",           "Use 300+ steps at 30Hz (10+ seconds)",                True),
    ("#9", "torch.load serialization",   "Crash on checkpoint load",       "add_safe_globals([NormalizationMode])",               False),
    ("#10", "Wrong import path",         "ModuleNotFoundError",            "so101_follower, not so_follower",                      False),
]

fig, ax = plt.subplots(figsize=(16, 6))
ax.axis('off')

# Table header
columns = ['#', 'Bug', 'Symptom', 'Fix', 'Silent?']
col_widths = [0.04, 0.16, 0.20, 0.42, 0.08]
col_positions = [0]
for w in col_widths[:-1]:
    col_positions.append(col_positions[-1] + w)

header_y = 0.95
for col, x_pos in zip(columns, col_positions):
    ax.text(x_pos, header_y, col, fontsize=11, fontweight='bold',
            color='white', va='top', transform=ax.transAxes,
            bbox=dict(boxstyle='round,pad=0.3', facecolor=ACCENT, alpha=0.9))

for row_i, (num, bug, symptom, fix, silent) in enumerate(bugs):
    y = header_y - 0.08 * (row_i + 1)
    row_color = '#FFF5F0' if row_i % 2 == 0 else BG
    texts = [num, bug, symptom, fix, 'Yes' if silent else 'No']
    for txt, x_pos in zip(texts, col_positions):
        color = RED if (txt == 'Yes') else (TEAL if txt == 'No' else '#333333')
        ax.text(x_pos, y, txt, fontsize=9, va='top', color=color,
                transform=ax.transAxes)

ax.set_title('The 10 Deployment Bugs — Complete Reference',
             fontsize=15, color=ACCENT, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Print as clean text table too
print(f"{'#':<4s} {'Bug':<25s} {'Symptom':<25s} {'Silent?':<8s}")
print(f"{'-'*4} {'-'*25} {'-'*25} {'-'*8}")
for num, bug, symptom, fix, silent in bugs:
    print(f"{num:<4s} {bug:<25s} {symptom:<25s} {'YES' if silent else 'NO':<8s}")
print(f"\n8 out of 10 bugs are SILENT — no error, no crash, just wrong behavior.")

In [ ]:
# DOs and DON'Ts
dos_and_donts = [
    ("DO",   "Use explicit joint ordering from your dataset config"),
    ("DON'T", "Use sorted() or dict iteration order for state vectors"),
    ("DO",   "Use a deque-based action queue for action chunking"),
    ("DON'T", "Regenerate the full action chunk every timestep"),
    ("DO",   "Cast inputs to model.dtype (bfloat16)"),
    ("DON'T", "Call .float() on pretrained bfloat16 models"),
    ("DO",   "Populate config.input_features from dataset metadata"),
    ("DON'T", "Assume from_pretrained() fills in features automatically"),
    ("DO",   "Add postprocessor (denormalization) before sending to robot"),
    ("DON'T", "Send raw normalized model outputs to motor controllers"),
    ("DO",   "Create an explicit camera rename map"),
    ("DON'T", "Assume camera names match between dataset and robot"),
    ("DO",   "Capture at full camera resolution (640x480+)"),
    ("DON'T", "Set camera capture resolution to 224x224 directly"),
    ("DO",   "Run episodes for 300+ steps at 30Hz (10+ seconds)"),
    ("DON'T", "Use 50 steps and wonder why the task never completes"),
    ("DO",   "Add NormalizationMode to torch safe globals"),
    ("DON'T", "Use weights_only=False as a blanket fix (security risk)"),
    ("DO",   "Double-check robot type: so101_follower / so101_leader"),
    ("DON'T", "Guess import paths — check lerobot source or docs"),
]

print("=" * 70)
print("  DOs and DON'Ts for Robot Deployment")
print("=" * 70)
for label, text in dos_and_donts:
    marker = '  +' if label == 'DO' else '  x'
    print(f"{marker} {label:>5s}: {text}")
print("=" * 70)

---

## Exercises

### Exercise 1: Bug Hunter

Each of the three code snippets below contains **2 bugs** from our list. Find all 6 bugs.

**Snippet A:**
```python
state = robot.read_state()
vec = np.array([state[k] for k in sorted(state.keys())])  # Bug ?
normed = (vec - means) / stds
action = model(torch.tensor(normed).float())  # Bug ?
robot.send(action.detach().numpy())
```

**Snippet B:**
```python
for step in range(50):  # Bug ?
    obs = get_camera_obs()
    chunk = policy.predict(obs)
    robot.send(chunk[0])  # Bug ?
```

**Snippet C:**
```python
cam = cv2.VideoCapture(0)
cam.set(cv2.CAP_PROP_FRAME_WIDTH, 224)   # Bug ?
cam.set(cv2.CAP_PROP_FRAME_HEIGHT, 224)
obs = {"observation.images.webcam": frame}  # Bug ? (model expects "camera1")
```

### Exercise 2: Build `validate_state_vector`

In [ ]:
# Exercise 2: Build a state vector validator
# Fill in the function below

def validate_state_vector(state_dict, expected_order):
    """Check that state_dict keys match expected_order exactly.

    Args:
        state_dict: dict mapping joint names to values
        expected_order: list of joint names in the correct order

    Returns:
        (is_valid, message) tuple
    """
    dict_keys = list(state_dict.keys())

    # Check 1: same set of keys?
    missing = set(expected_order) - set(dict_keys)
    extra = set(dict_keys) - set(expected_order)
    if missing:
        return False, f"Missing joints: {missing}"
    if extra:
        return False, f"Unexpected joints: {extra}"

    # Check 2: same order?
    if dict_keys != expected_order:
        # Find the first mismatch
        for i, (got, expected) in enumerate(zip(dict_keys, expected_order)):
            if got != expected:
                return False, f"Order mismatch at index {i}: got '{got}', expected '{expected}'"

    return True, "VALID: all joints present in correct order"


# Test it
good_state = OrderedDict([(n, 0.0) for n in JOINT_NAMES])
bad_state = OrderedDict([(n, 0.0) for n in sorted(JOINT_NAMES)])

valid, msg = validate_state_vector(good_state, JOINT_NAMES)
print(f"Good state: valid={valid}, {msg}")

valid, msg = validate_state_vector(bad_state, JOINT_NAMES)
print(f"Bad state:  valid={valid}, {msg}")

### Exercise 3: Build `diagnose_deployment`

In [ ]:
# Exercise 3: Build a deployment diagnostic tool
# Fill in the missing checks

def diagnose_deployment(config):
    """Check a deployment config for all 10 common bugs.

    Args:
        config: dict with keys like 'joint_order', 'use_action_queue',
                'model_dtype', 'features_populated', 'has_postprocessor',
                'camera_rename_map', 'capture_resolution', 'n_steps',
                'fps', 'safe_globals_registered', 'robot_type'
    """
    results = []

    # Bug #1: State vector order
    if config.get('joint_order') == sorted(config.get('joint_order', [])):
        joint_list = config.get('joint_order', [])
        if joint_list and joint_list != JOINT_NAMES:
            results.append(('FAIL', '#1 State vector order', 'Joints appear alphabetically sorted'))
        else:
            results.append(('PASS', '#1 State vector order', 'OK'))
    else:
        results.append(('PASS', '#1 State vector order', 'Not alphabetically sorted'))

    # Bug #2: Action chunking
    if config.get('use_action_queue', False):
        results.append(('PASS', '#2 Action chunking', 'Using action queue'))
    else:
        results.append(('FAIL', '#2 Action chunking', 'No action queue — will be jerky'))

    # Bug #3: .float()
    if config.get('model_dtype') == 'bfloat16' and config.get('cast_model_to_float32', False):
        results.append(('FAIL', '#3 .float() conversion', 'Model will be cast to float32'))
    else:
        results.append(('PASS', '#3 .float() conversion', 'OK'))

    # Bug #4: Features populated
    if config.get('features_populated', False):
        results.append(('PASS', '#4 Features populated', 'Features set from dataset'))
    else:
        results.append(('FAIL', '#4 Features populated', 'Features dict is empty!'))

    # Bug #5: Postprocessor
    if config.get('has_postprocessor', False):
        results.append(('PASS', '#5 Postprocessor', 'Denormalization enabled'))
    else:
        results.append(('FAIL', '#5 Postprocessor', 'Raw normalized values will go to motors!'))

    # Bug #6: Camera rename map
    if config.get('camera_rename_map'):
        results.append(('PASS', '#6 Camera rename map', 'Mapping defined'))
    else:
        results.append(('FAIL', '#6 Camera rename map', 'No mapping — cameras may be black/swapped'))

    # Bug #7: Resolution
    cap_res = config.get('capture_resolution', (224, 224))
    if cap_res[0] >= 480 and cap_res[1] >= 640:
        results.append(('PASS', '#7 Resolution', f'Capturing at {cap_res[1]}x{cap_res[0]}'))
    else:
        results.append(('FAIL', '#7 Resolution', f'Capturing at {cap_res[1]}x{cap_res[0]} — too small!'))

    # Bug #8: Episode length
    n_steps = config.get('n_steps', 50)
    fps = config.get('fps', 30)
    duration = n_steps / fps
    if duration >= 8.0:
        results.append(('PASS', '#8 Episode length', f'{n_steps} steps = {duration:.1f}s'))
    else:
        results.append(('FAIL', '#8 Episode length', f'{n_steps} steps = {duration:.1f}s — too short!'))

    # Bug #9: Safe globals
    if config.get('safe_globals_registered', False):
        results.append(('PASS', '#9 Safe globals', 'NormalizationMode registered'))
    else:
        results.append(('WARN', '#9 Safe globals', 'May crash on torch.load()'))

    # Bug #10: Robot type
    robot_type = config.get('robot_type', '')
    if 'so101' in robot_type:
        results.append(('PASS', '#10 Robot type', f'Using "{robot_type}"'))
    elif 'so_' in robot_type:
        results.append(('FAIL', '#10 Robot type', f'"{robot_type}" should be "so101_..."'))
    else:
        results.append(('PASS', '#10 Robot type', f'Using "{robot_type}"'))

    # Print results
    print("=" * 65)
    print("  DEPLOYMENT DIAGNOSTIC REPORT")
    print("=" * 65)
    n_pass = sum(1 for r in results if r[0] == 'PASS')
    n_fail = sum(1 for r in results if r[0] == 'FAIL')
    n_warn = sum(1 for r in results if r[0] == 'WARN')
    for status, bug, detail in results:
        icon = '+' if status == 'PASS' else ('x' if status == 'FAIL' else '?')
        print(f"  [{icon}] {status:<4s} {bug:<28s} {detail}")
    print("=" * 65)
    print(f"  Results: {n_pass} passed, {n_fail} failed, {n_warn} warnings")
    if n_fail == 0:
        print("  DEPLOYMENT READY")
    else:
        print(f"  FIX {n_fail} ISSUE(S) BEFORE DEPLOYING")
    print("=" * 65)
    return results


# Test with a BUGGY config
print("--- BUGGY CONFIG ---")
buggy_config = {
    'joint_order': sorted(JOINT_NAMES),  # Bug #1
    'use_action_queue': False,            # Bug #2
    'model_dtype': 'bfloat16',
    'cast_model_to_float32': True,        # Bug #3
    'features_populated': False,          # Bug #4
    'has_postprocessor': False,           # Bug #5
    'camera_rename_map': None,            # Bug #6
    'capture_resolution': (224, 224),     # Bug #7
    'n_steps': 50,                        # Bug #8
    'fps': 30,
    'safe_globals_registered': False,     # Bug #9
    'robot_type': 'so_follower',          # Bug #10
}
_ = diagnose_deployment(buggy_config)

In [ ]:
# Test with a FIXED config
print("--- FIXED CONFIG ---")
fixed_config = {
    'joint_order': JOINT_NAMES,
    'use_action_queue': True,
    'model_dtype': 'bfloat16',
    'cast_model_to_float32': False,
    'features_populated': True,
    'has_postprocessor': True,
    'camera_rename_map': {'webcam': 'camera1', 'arm_cam': 'camera2'},
    'capture_resolution': (480, 640),
    'n_steps': 300,
    'fps': 30,
    'safe_globals_registered': True,
    'robot_type': 'so101_follower',
}
_ = diagnose_deployment(fixed_config)

### Exercise 4: Action Horizon Mismatch

Your model predicts chunks of 50 actions, but your code only handles chunks of 16. Simulate this bug and fix it.

In [ ]:
# Exercise 4: Action horizon mismatch
# The model outputs 50 actions but the queue only takes 16

def model_predict_50(step):
    """Model that outputs 50-step action chunks."""
    t = np.linspace(step * 0.02, step * 0.02 + 50 * 0.02, 50)
    return np.sin(t) * 30.0 + np.random.randn(50) * 0.5


def control_loop_mismatch(n_steps=200, handled_horizon=16):
    """BUG: only handles 16 out of 50 predicted actions."""
    trajectory = []
    queue = deque()
    model_calls = 0
    wasted_actions = 0
    for step in range(n_steps):
        if len(queue) == 0:
            chunk = model_predict_50(step)
            model_calls += 1
            # BUG: only use first 16 actions, waste the rest
            queue.extend(chunk[:handled_horizon])
            wasted_actions += len(chunk) - handled_horizon
        trajectory.append(queue.popleft())
    return np.array(trajectory), model_calls, wasted_actions


def control_loop_full_horizon(n_steps=200):
    """FIXED: use all 50 predicted actions."""
    trajectory = []
    queue = deque()
    model_calls = 0
    for step in range(n_steps):
        if len(queue) == 0:
            chunk = model_predict_50(step)
            model_calls += 1
            queue.extend(chunk)  # use ALL 50 actions
        trajectory.append(queue.popleft())
    return np.array(trajectory), model_calls, 0


np.random.seed(42)
traj_16, calls_16, wasted_16 = control_loop_mismatch(200, 16)
np.random.seed(42)
traj_50, calls_50, wasted_50 = control_loop_full_horizon(200)

print(f"Horizon=16: {calls_16} model calls, {wasted_16} wasted actions")
print(f"Horizon=50: {calls_50} model calls, {wasted_50} wasted actions")
print(f"\nUsing only 16/50 actions wastes {wasted_16/(calls_16*50)*100:.0f}% of compute!")
print(f"And calls the model {calls_16/calls_50:.1f}x more often.")

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(traj_16, color=RED, linewidth=1.5, label=f'Horizon=16 ({calls_16} calls)', alpha=0.8)
ax.plot(traj_50, color=TEAL, linewidth=1.5, label=f'Horizon=50 ({calls_50} calls)', alpha=0.8)
ax.set_xlabel('Timestep', fontweight='bold')
ax.set_ylabel('Joint angle (deg)', fontweight='bold')
ax.set_title('Action Horizon Mismatch: 16 vs 50', fontsize=13, color=ACCENT, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

### Exercise 5 (Challenge): Deployment Readiness Checker

Build a complete deployment readiness checker that takes a config dict and validates: joint order, camera names, resolution, episode length, postprocessor presence, and prints pass/fail for each. Extend `diagnose_deployment` above to also:
- Verify that camera images are non-zero
- Verify that state vector values are within physically reasonable ranges
- Compute and display the total estimated episode duration
- Check that the action dimension matches the state dimension

In [ ]:
# Exercise 5 (Challenge): Complete deployment readiness checker
# Extend the diagnose_deployment function above.
# Add checks for:
#   - Camera images are non-zero
#   - State values within physical range (e.g., joint angles in [-180, 180])
#   - Action dimension == state dimension
#   - Estimated episode duration displayed

def deployment_readiness_check(config, sample_obs=None):
    """
    Full deployment readiness checker.

    Args:
        config: deployment config dict (same as diagnose_deployment)
        sample_obs: optional dict with sample observations to validate
            e.g. {'state': np.array([...]), 'images': {'webcam': np.array(...)}}
    """
    # Start with the base 10-bug check
    # YOUR CODE HERE: call diagnose_deployment and extend with additional checks
    pass


print("Implement deployment_readiness_check() above!")
print("It should combine diagnose_deployment() with image, range, and dimension checks.")